# EDA — IEEE-CIS Fraud Detection

**This notebook analyses; it does not define.** Every function it calls lives in `src/`, so nothing here is reusable-logic-trapped-in-a-notebook. Each section ends with the feature-engineering or validation decision it produced, and those decisions are implemented in `src/features/` and `src/data/`.

Scope is deliberately narrow — a handful of questions that change the design, rather than a gallery of plots. The exhaustive per-column profile already exists in `reports/dataset_audit.md`, generated by `scripts/inspect_dataset.py`.

**Prerequisite:** `python scripts/build_dataset.py`

Questions:
1. How imbalanced is the target, and is prevalence stable over time?
2. Is the missingness structural (a signal) or random (a defect)?
3. Does transaction amount separate the classes on its own?
4. Do temporal patterns exist that cyclical features can capture?
5. Does identity coverage carry signal?
6. How far does the train distribution move from the later test period?

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.data.loading import joined_path, load_parquet
from src.data.schema import C_COLUMNS, D_COLUMNS, M_COLUMNS, SECONDS_PER_DAY, TARGET, V_COLUMNS
from src.data.validation import missing_value_profile
from src.utils.paths import FIGURES_DIR, PROCESSED_DIR

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 60)

print("Reading the joined training frame (this is ~1 GB in memory)")
df = load_parquet(joined_path("train"))
print(df.shape)

## 1. Class imbalance, and whether it holds still

The headline imbalance number is well known for this dataset. The question that actually matters for the *design* is whether prevalence is stationary — because if it is not, random cross-validation folds are not exchangeable samples.

In [ ]:
counts = df[TARGET].value_counts()
prevalence = df[TARGET].mean()
print(f"legit  : {counts[0]:>7,}")
print(f"fraud  : {counts[1]:>7,}")
print(f"prevalence      : {prevalence:.4%}")
print(f"imbalance ratio : {counts[0] / counts[1]:.2f} : 1")
print(f"\nAccuracy of a 'never fraud' predictor: {1 - prevalence:.4%}")
print("=> accuracy is unusable as a metric; PR-AUC is the selection metric.")
print(f"=> PR-AUC no-skill baseline is the prevalence itself: {prevalence:.4f}")

In [ ]:
df["day"] = df["TransactionDT"] // SECONDS_PER_DAY
daily = df.groupby("day")[TARGET].agg(["size", "mean"])

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(daily.index, daily["size"], lw=1, color="#1d4ed8")
axes[0].set_ylabel("transactions / day")
axes[0].set_title("Volume and fraud rate over the 182-day training period")

axes[1].plot(daily.index, daily["mean"] * 100, lw=1, color="#c2410c")
axes[1].axhline(prevalence * 100, ls="--", color="#64748b", label=f"overall {prevalence:.2%}")
axes[1].set_ylabel("fraud rate (%)")
axes[1].set_xlabel("day index")
axes[1].legend()
plt.tight_layout()
plt.show()

blocks = df.groupby(df["day"] // 30)[TARGET].agg(["size", "mean"])
blocks["mean"] = (blocks["mean"] * 100).round(4)
print("Fraud rate per 30-day block (%):")
print(blocks)

**Decision.** Prevalence ranges 2.48%–4.18% across 30-day blocks, so it is *not* stationary. Combined with the disjoint train/test periods, this rules out random stratified K-fold. → implemented as `PurgedForwardChainingCV` in `src/data/splitting.py`, and imbalance is handled by reweighting (`scale_pos_weight`) rather than resampling.

## 2. Is missingness a signal or a defect?

This determines whether to impute at all. If missingness is random, imputation is harmless; if it is structural, imputing destroys information.

In [ ]:
profile = missing_value_profile(df.drop(columns=["day"]))

families = {
    "C (counting)": C_COLUMNS,
    "D (timedelta)": D_COLUMNS,
    "M (match flags)": M_COLUMNS,
    "V (Vesta engineered)": V_COLUMNS,
}
summary = []
for name, columns in families.items():
    present = [c for c in columns if c in df.columns]
    rates = profile.set_index("column").loc[present, "missing_rate"]
    summary.append({
        "family": name, "n_columns": len(present),
        "min": f"{rates.min():.3%}", "mean": f"{rates.mean():.3%}", "max": f"{rates.max():.3%}",
        "n_distinct_rates": rates.round(6).nunique(),
    })
print(pd.DataFrame(summary).to_string(index=False))
print("\nNote 'n_distinct_rates': if 339 V columns share only a handful of distinct")
print("missing rates, they appear and vanish in blocks -> structural, not random.")

In [ ]:
# Does 'how much do we know about this transaction' predict fraud?
df["n_missing"] = df.isna().sum(axis=1)
bins = pd.qcut(df["n_missing"], q=8, duplicates="drop")
by_missing = df.groupby(bins, observed=True)[TARGET].agg(["size", "mean"])
by_missing["fraud_rate_%"] = (by_missing["mean"] * 100).round(3)
print(by_missing[["size", "fraud_rate_%"]])

ax = (by_missing["mean"] * 100).plot(kind="bar", figsize=(10, 4), color="#c2410c")
ax.axhline(prevalence * 100, ls="--", color="#64748b")
ax.set_ylabel("fraud rate (%)")
ax.set_xlabel("missing fields per row (binned)")
ax.set_title("Record completeness vs fraud rate")
plt.tight_layout(); plt.show()

**Decision.** Missingness is structural (C columns never missing; V columns share block-level rates) and it correlates with the target. → LightGBM receives **native NaN**, never imputation. Missingness is promoted to explicit features (`n_missing_total`, per-family counts) in `src/features/builders.py::add_missingness_features`. Only the linear baseline is imputed, and it gets `_isna` indicator columns so the comparison stays fair.

## 3. Does the amount separate the classes?

In [ ]:
print(df.groupby(TARGET)["TransactionAmt"].describe()[["count", "mean", "50%", "75%", "max"]])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for label, colour in [(0, "#1d4ed8"), (1, "#c2410c")]:
    subset = np.log1p(df.loc[df[TARGET] == label, "TransactionAmt"])
    axes[0].hist(subset, bins=60, alpha=0.6, density=True,
                 label=f"isFraud={label}", color=colour)
axes[0].set_xlabel("log1p(TransactionAmt)"); axes[0].set_ylabel("density")
axes[0].set_title("Amount distribution by class (log scale)"); axes[0].legend()

# The decimal part: card testing and currency conversion leave fingerprints here.
cents = (df["TransactionAmt"] - np.floor(df["TransactionAmt"])).round(4)
is_round = (cents == 0)
rates = [df.loc[is_round, TARGET].mean() * 100, df.loc[~is_round, TARGET].mean() * 100]
axes[1].bar(["round amount", "has decimals"], rates, color=["#1d4ed8", "#c2410c"])
axes[1].axhline(prevalence * 100, ls="--", color="#64748b")
axes[1].set_ylabel("fraud rate (%)"); axes[1].set_title("Fraud rate by amount decimal part")
for i, value in enumerate(rates):
    axes[1].text(i, value, f"{value:.2f}%", ha="center", va="bottom")
plt.tight_layout(); plt.show()

**Decision.** The class means are close (≈149 vs ≈135), so *absolute* amount is weak on its own — but the distribution is heavily right-skewed and the decimal part clearly separates. → `log_amount` (required by the linear baseline), `amount_cents`, `is_round_amount`, and crucially the **per-entity** deviation features (`entity_card_amt_zscore`, `..._amt_diff_from_mean`): fraud is anomalous *relative to an account's own history*, not in absolute terms.

## 4. Temporal patterns that cyclical features can capture

Absolute time is unusable (train and test ranges are disjoint), so the only question is whether *cyclical* time carries signal.

In [ ]:
df["hour"] = (df["TransactionDT"] // 3600) % 24
df["dow"] = (df["TransactionDT"] // SECONDS_PER_DAY) % 7

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
hourly = df.groupby("hour")[TARGET].agg(["size", "mean"])
axes[0].bar(hourly.index, hourly["mean"] * 100, color="#c2410c", alpha=0.85)
axes[0].axhline(prevalence * 100, ls="--", color="#64748b")
twin = axes[0].twinx()
twin.plot(hourly.index, hourly["size"], color="#1d4ed8", lw=1.5)
twin.set_ylabel("volume", color="#1d4ed8"); twin.grid(False)
axes[0].set_xlabel("hour of day"); axes[0].set_ylabel("fraud rate (%)")
axes[0].set_title("Fraud rate and volume by hour")

daily_dow = df.groupby("dow")[TARGET].mean() * 100
axes[1].bar(daily_dow.index, daily_dow.values, color="#c2410c", alpha=0.85)
axes[1].axhline(prevalence * 100, ls="--", color="#64748b")
axes[1].set_xlabel("day of week"); axes[1].set_ylabel("fraud rate (%)")
axes[1].set_title("Fraud rate by day of week")
plt.tight_layout(); plt.show()

print(f"Fraud rate range across hours: {hourly['mean'].min():.2%} - {hourly['mean'].max():.2%}")
print("Volume trough coinciding with a fraud-rate peak = automated activity at off-peak hours.")

**Decision.** Fraud rate varies strongly by hour, and the peak coincides with the volume trough — consistent with automated card testing when review staffing is thin. → `hour_of_day`, `day_of_week`, `is_night`, `is_weekend` (cyclical, so they transfer across the 30-day gap). Absolute day index is used **only** to build folds and `D`-column anchors, then dropped.

## 5. Identity coverage, email and device signals

In [ ]:
print(f"identity coverage: {df['identity_present'].mean():.4%}")
print(df.groupby("identity_present")[TARGET].agg(["size", "mean"]).rename(
    columns={"mean": "fraud_rate"}))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

top_email = df["P_emaildomain"].value_counts().head(10).index
email_rates = (df[df["P_emaildomain"].isin(top_email)]
               .groupby("P_emaildomain", observed=True)[TARGET].mean().sort_values() * 100)
axes[0].barh(email_rates.index.astype(str), email_rates.values, color="#c2410c")
axes[0].axvline(prevalence * 100, ls="--", color="#64748b")
axes[0].set_title("Fraud rate by purchaser email domain"); axes[0].set_xlabel("fraud rate (%)")

device_rates = df.groupby("DeviceType", observed=True)[TARGET].mean().sort_values() * 100
axes[1].bar(device_rates.index.astype(str), device_rates.values, color="#1d4ed8")
axes[1].axhline(prevalence * 100, ls="--", color="#64748b")
axes[1].set_title("Fraud rate by device type"); axes[1].set_ylabel("fraud rate (%)")

product_rates = df.groupby("ProductCD", observed=True)[TARGET].mean().sort_values() * 100
axes[2].bar(product_rates.index.astype(str), product_rates.values, color="#047857")
axes[2].axhline(prevalence * 100, ls="--", color="#64748b")
axes[2].set_title("Fraud rate by ProductCD"); axes[2].set_ylabel("fraud rate (%)")
plt.tight_layout(); plt.show()

print(f"\nDeviceInfo distinct values: {df['DeviceInfo'].nunique():,}")
print("=> too high-cardinality to use verbatim; parse to a vendor token instead.")

**Decision.** Whether identity resolution succeeded is itself informative, and `ProductCD` separates strongly. `DeviceInfo` has >1,000 distinct values and would overfit verbatim. → `identity_present`, `n_missing_id`, `device_vendor` (parsed token), `os_family`/`os_version_major`, `browser_family`/`browser_version_major`, screen dimensions from `id_33`, plus `email_domains_match` and provider/TLD splits. High-cardinality identifiers get **frequency encoding fit on the training partition only**.

## 6. Train vs the later test period

The Kaggle test set is unlabeled, so it cannot be scored — but its *features* can be compared, and the shift is real: it begins 30 days after training ends.

In [ ]:
test_path = joined_path("test")
if not test_path.is_file():
    print("Test split not built. Run: python scripts/build_dataset.py --with-test")
else:
    compare_columns = ["TransactionDT", "TransactionAmt", "D13", "V45", "C13", "dist1"]
    test_df = load_parquet(test_path, columns=[c for c in compare_columns if c != "day"])

    print(f"train days: {df['day'].min()}-{df['day'].max()}")
    test_day = test_df['TransactionDT'] // SECONDS_PER_DAY
    print(f"test  days: {test_day.min()}-{test_day.max()}")
    print(f"gap       : {(test_df['TransactionDT'].min() - df['TransactionDT'].max()) / SECONDS_PER_DAY:.1f} days\n")

    rows = []
    for column in compare_columns[1:]:
        if column in df.columns and column in test_df.columns:
            rows.append({
                "feature": column,
                "train_missing": f"{df[column].isna().mean():.3%}",
                "test_missing": f"{test_df[column].isna().mean():.3%}",
                "train_mean": round(float(df[column].mean()), 3),
                "test_mean": round(float(test_df[column].mean()), 3),
            })
    print(pd.DataFrame(rows).to_string(index=False))
    print("\nFor the full PSI/KS drift report across all features:")
    print("  python scripts/monitor.py --current test")

**Decision.** The shift is measurable (e.g. `D13` missingness 89.5% → 75.6%, the `V39`–`V52` block 28.6% → 15.2%). → this is the reference comparison used by `src/monitoring/drift.py`, and the reason the monitoring demo uses the real test period rather than synthetic perturbations.

---

## Summary of decisions this notebook produced

| finding | decision | implemented in |
|---|---|---|
| Prevalence non-stationary (2.48%–4.18%) | time-aware CV, not random K-fold | `src/data/splitting.py` |
| 27.58:1 imbalance | PR-AUC as selection metric; reweighting not SMOTE | `src/evaluation/metrics.py`, `src/models/estimators.py` |
| Missingness structural and predictive | native NaN for trees; missingness as features | `src/features/builders.py` |
| Amount weak absolutely, informative relatively | per-entity deviation features | `src/features/aggregations.py` |
| Strong hour-of-day pattern | cyclical time features only | `src/features/builders.py` |
| Identity present for only ~24% | `identity_present` as an explicit feature | `src/data/loading.py` |
| `DeviceInfo` >1,000 distinct values | parse to vendor; frequency-encode | `src/features/builders.py` |
| Real train→test covariate shift | drift monitoring against the test period | `src/monitoring/drift.py` |

Next: `python scripts/train.py`